# Jupyter kernels from the CUDA-Q container

Run the following 2 cells if you are using a Jupyter notebook from OnDemand. Make sure you have selected the Python kernel, **Python 3 (ipykernel)**, in the top-right corner.

In [ ]:
# check version of ipykernel and cudaq container
!apptainer exec /ocean/containers/quantum/cuda-quantum_latest.sif python3 -c "import ipykernel, cudaq; print(ipykernel.__version__, cudaq.__version__)"

In [ ]:
import json, os, shutil, subprocess

sif    = "/ocean/containers/quantum/cuda-quantum_latest.sif"
of_pkg = "/usr/local/lib/python3.12/dist-packages/openfermion/testing/data"
of_dir = f"{os.environ['PROJECT']}/CUDAQ/openfermion_data"
runner = shutil.which("apptainer")
assert runner and os.path.exists(sif)

# Copy OpenFermion's bundled molecule files into a writable folder (once)
os.makedirs(of_dir, exist_ok=True)
subprocess.run([runner, "exec", "--bind", of_dir, sif,
                "cp", "-r", "--update=none", f"{of_pkg}/.", of_dir])

for name, display, flags in [("cudaq-cpu", "Python (CUDA-Q CPU)", []),
                             ("cudaq-gpu", "Python (CUDA-Q GPU)", ["--nv"])]:
    kdir = os.path.expanduser(f"~/.local/share/jupyter/kernels/{name}")
    os.makedirs(kdir, exist_ok=True)
    argv = [runner, "exec", "--cleanenv", *flags,
            "--bind", "/ocean",
            "--bind", f"{of_dir}:{of_pkg}",
            "--env", "PYTHONNOUSERSITE=1",
            sif, "python3", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
    with open(f"{kdir}/kernel.json", "w") as f:
        json.dump({"argv": argv, "display_name": display, "language": "python"}, f, indent=2)
    print("created", name)

From now on, select the Python kernel in the top-right corner: choose **Python (CUDA-Q CPU)** for CPU, or **Python (CUDA-Q GPU)** for GPU.

(Refresh the browser page first if the new kernels don't appear in the list.)